# Clase 5 · Notebook 01
# Regresión simple y funciones de pérdida

**Introducción al Deep Learning — Módulo 2, Unidad 3: Regresión**

La regresión predice un **valor numérico continuo** (no una clase). En este cuaderno construiremos una
**regresión lineal simple** con una red de una sola neurona y aprenderemos a medirla con las métricas de
regresión: **MSE, RMSE, MAE y R²**.

Usaremos el dataset **diabetes** de Scikit-Learn y, como variable independiente, el índice de masa corporal (BMI).

1. Preparar los datos (una variable).
2. Construir la red: 1 neurona con activación lineal.
3. Entrenar y visualizar la recta de regresión.
4. Evaluar con MSE, RMSE, MAE y R².

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Preparar los datos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
tf.random.set_seed(42); np.random.seed(42)

datos = load_diabetes()
# Variable independiente: BMI (columna 2). Variable dependiente: progresión de la enfermedad.
bmi_idx = list(datos.feature_names).index("bmi")
X = datos.data[:, bmi_idx].reshape(-1, 1).astype("float32")
y = datos.target.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Variable independiente: BMI | Variable dependiente: progresión")
print("Entrenamiento:", X_train.shape[0], "| Test:", X_test.shape[0])

## 2. Construir la red

Para una regresión lineal simple basta **una neurona con activación lineal**: aprende la pendiente y la
ordenada (2 coeficientes). Pérdida y métricas propias de regresión: **MSE** y **MAE**.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

modelo = Sequential([
    Input(shape=(1,)),
    Dense(1, activation="linear"),
])
modelo.compile(optimizer="adam", loss="mse", metrics=["mae"])
modelo.summary()

## 3. Entrenar y visualizar la recta

In [ ]:
historia = modelo.fit(X_train, y_train, epochs=150, batch_size=16,
                      validation_split=0.1, verbose=0)
print("Entrenamiento terminado.")
w, b = modelo.layers[0].get_weights()
print(f"Recta aprendida:  y = {w[0,0]:.1f} * x + {b[0]:.1f}")

In [ ]:
import matplotlib.pyplot as plt

xs = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
ys = modelo.predict(xs, verbose=0)

plt.figure(figsize=(7, 5))
plt.scatter(X_test, y_test, color="#4355FF", alpha=0.6, label="datos (test)")
plt.plot(xs, ys, color="#FF647E", lw=3, label="recta de regresión")
plt.xlabel("BMI (normalizado)"); plt.ylabel("Progresión de la enfermedad")
plt.title("Regresión lineal simple"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Evaluar: MSE, RMSE, MAE y R²

Calculamos las métricas sobre el conjunto de **test**.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = modelo.predict(X_test, verbose=0).ravel()

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"MSE  : {mse:.1f}   (penaliza mucho los errores grandes)")
print(f"RMSE : {rmse:.1f}   (misma escala que la variable objetivo)")
print(f"MAE  : {mae:.1f}   (error medio absoluto)")
print(f"R2   : {r2:.3f}   (0 = malo, 1 = ajuste perfecto)")

Con una sola variable (BMI) el R² es moderado: el modelo capta la tendencia general, pero le **falta
información** para predecir bien. Esto motiva la **regresión múltiple** del Notebook 02.

## Resumen

- La regresión predice un **número continuo**; una neurona con activación **lineal** = recta de regresión.
- Pérdidas de regresión: **MSE** (sensible a outliers), **RMSE** (escala de los datos), **MAE** (robusta).
- **R²** mide la calidad del ajuste sin depender de la escala (hasta 1).
- Con una sola variable el ajuste suele quedarse corto.

➡️ En el **Notebook 02** usaremos todas las variables (regresión múltiple) y añadiremos **regularización**.